[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/duoan/TorchCode/blob/master/templates/12_linear_attention.ipynb)

# 🔴 Hard: Linear Self-Attention

Реализуйте **Linear Attention** — вариант attention, который меняет порядок матричных произведений и избегает явной `(S,S)` матрицы. При `S` намного больше `D` это даёт $O(SD^2)$ вместо $O(S^2D)$.

> Это не быстрый способ вычислить обычный softmax attention, а другой механизм. Feature map приближает/заменяет similarity kernel, поэтому его численные ответы не обязаны совпадать с softmax attention.

Replace softmax with a **kernel feature map** $\phi$:

$$\text{LinearAttn}(Q,K,V) = \frac{\phi(Q) \left(\phi(K)^T V\right)}{\phi(Q) \cdot \sum \phi(K)}$$

### Feature map
Use $\phi(x) = \text{elu}(x) + 1$ (ensures non-negative features). Положительность нужна, чтобы знаменатель можно было интерпретировать как нормализацию и он не сокращался из-за взаимного уничтожения знаков.

## Почему произведения можно переставить

Для softmax нормализация каждой пары Q/K мешает простой ассоциативной перестановке. После feature map ненормированный числитель имеет вид `Qφ @ (Kφᵀ @ V)`. Сначала вычисляется summary `KV` формы `(B,D_k,D_v)`, затем каждая позиция Q читает его. Отдельно `sum(Kφ, seq)` формы `(B,D_k)` даёт знаменатель для каждой Q-позиции.

Для `Q,K=(2,1000,64)` и `V=(2,1000,32)` не создаётся `(2,1000,1000)`. Вместо неё основное промежуточное состояние — `(2,64,32)`, output остаётся `(2,1000,32)`. В знаменатель добавляется маленький `eps` для устойчивости.

### Signature
```python
def linear_attention(Q, K, V):
    # Q: (B, S, D_k), K: (B, S, D_k), V: (B, S, D_v)
    # Returns: (B, S, D_v)
```

### Key insight
Instead of computing the $S \times S$ attention matrix, compute $\phi(K)^T V$ first (a $D_k \times D_v$ matrix), then multiply by $\phi(Q)$.

### Rules
- Must use a feature map (NOT softmax)
- Must be O(S·D²) — should run fast on long sequences
- You **may** use `F.elu`

> Формула упражнения не causal: `KᵀV` суммирует всю последовательность. Causal linear attention требует prefix sums состояний K и KV, чтобы позиция не получила будущую информацию.

## План реализации

1. Примените `elu(x)+1` к Q и K.
2. Сверните sequence-ось, получив `KφᵀV`.
3. Умножьте Qφ на summary для числителя.
4. Получите поэлементный знаменатель Qφ против суммы Kφ и нормализуйте с `eps`.

## Быстрые проверки

- Output shape равна `(B,S,D_v)`.
- Реализация работает при `D_v != D_k`.
- Нет тензора, две оси которого обе равны `S`.
- Для длинной последовательности runtime растёт примерно линейно по S при фиксированных D.

## Частые ошибки

- Забыть feature map у Q или K.
- Нормализовать одним scalar вместо отдельного знаменателя для каждой `(B,S)` позиции.
- Перепутать `D_k` и `D_v` в einsum/matmul.
- Утверждать, что результат должен совпасть с softmax attention.

## Материалы для глубокого изучения

- [Transformers are RNNs: Fast Autoregressive Transformers with Linear Attention](https://arxiv.org/abs/2006.16236) — исходная формулировка feature-map attention.
- [PMLR version with paper and supplementary material](https://proceedings.mlr.press/v119/katharopoulos20a.html) — статья и дополнительные материалы.
- [PyTorch einsum](https://docs.pytorch.org/docs/stable/generated/torch.einsum.html) — удобная запись свёрток по именованным индексам.

## Вопросы для самопроверки

1. Как ассоциативность убирает матрицу `(S,S)`?
2. Почему эта формула не равна softmax attention?
3. Что нужно изменить для causal режима?

In [ ]:
# Install torch-judge in Colab (no-op in JupyterLab/Docker)
try:
    import google.colab
    get_ipython().run_line_magic('pip', 'install -q torch-judge')
except ImportError:
    pass


In [ ]:
import torch
import torch.nn.functional as F

In [ ]:
# ✏️ YOUR IMPLEMENTATION HERE

def linear_attention(Q, K, V):
    pass  # Replace this

In [ ]:
# 🧪 Debug
Q = torch.randn(1, 8, 16)
K = torch.randn(1, 8, 16)
V = torch.randn(1, 8, 32)
out = linear_attention(Q, K, V)
print("Output shape:", out.shape)   # (1, 8, 32)
print("Has NaN?", torch.isnan(out).any().item())

In [ ]:
from torch_judge import check
check('linear_attention')